<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Ranked Actions and Reason Codes

To convert model output probabilities into actionable SEO workflows, recommendations are grouped into four distinct action archetypes:

| Priority Rank | Reason Code | Action Label | Trigger Archetype & Criteria | Target Outcome |
| :--- | :--- | :--- | :--- | :--- |
| **1** | `HIGH_IMPRESSION_LOW_CTR` | `REWRITE_TITLE_AND_META` | Page 1/2 rankings (Positions 4–20) with high impressions but low CTR relative to position baseline. | Immediate CTR boost without ranking changes. |
| **2** | `POSITION_DECAY_STALE` | `REFRESH_CONTENT` | Historical impression drop with average position drifting beyond position 10 over 30 days. | Regain lost SERP relevance and rankings. |
| **3** | `PAGE2_HIGH_POTENTIAL` | `ADD_INTERNAL_LINKS` | Striking-distance URLs (Positions 11–20) with solid engagement but low authority. | Push page from Page 2 onto Page 1. |
| **4** | `INTENT_MISMATCH_AUDIT` | `AUDIT_SEARCH_INTENT` | High GA4 sessions but minimal GSC search impressions/clicks (traffic driven by non-search sources). | Align content structure with search intent. |

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 2. Intended Use and Operational Limits

* **Intended Users:** Content strategists, SEO managers, and editorial teams.
* **Operational Scope:** A batch, offline decision-support queue intended for weekly or monthly content review cycles.

**Explicit Operational Limits:**
1. **Non-Realtime:** Designed for multi-day/monthly aggregations; does not handle real-time SERP volatility or daily rank tracking spikes.
2. **No Automated Publishing:** The playbook outputs prioritization flags; it does not generate or push live content changes automatically.
3. **No Penalty Recovery:** Cannot diagnose or remediate manual action penalties, algorithmic spam updates, or major site-wide technical crawl failures.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 3. Human Review Protocol & The Strict No-Go List

#### Human Pre-Action Verification Checklist
Before executing any flagged recommendation, an editor must verify:
1. **Search Intent Match:** Does the current ranking snippet match user intent, or is Google displaying a featured snippet/zero-click element?
2. **Brand / Legal Compliance:** Ensure title tag changes do not remove mandatory brand modifiers or legal disclaimers.
3. **Conversion Value:** Confirm the page carries business value (e.g., high-converting lead page vs. obsolete announcement).

---

#### The Strict No-Go List (Never Automate)
* **NO-GO 1: Bulk Automated Page Deletion / Unpublishing:** Never automatically delete or `404` pages based on low predicted scores.
* **NO-GO 2: Unreviewed AI Title/Meta Injecting:** Never automatically overwrite metadata without human editorial approval.
* **NO-GO 3: Automated 301/302 Redirect Chains:** Never automatically redirect URLs; redirecting high-traffic URLs without technical audit destroys link equity.
* **NO-GO 4: Automated Changes to Core Brand/Product Pages:** High-revenue transactional pages are strictly exempt from automated recommendation queues.

In [4]:
import duckdb
import os
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Authenticate DuckDB with HF Secret Token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Extract and assign human review queue
df_review = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) AS total_clicks,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS ctr
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 1000
    LIMIT 10
""").df()

# Audit simulation: flag items requiring human inspection
df_review['requires_human_signoff'] = True
df_review['flag_reason'] = np.where(
    df_review['avg_position'] <= 10,
    'PAGE_1_CTR_OPTIMIZATION',
    'STRIKING_DISTANCE_BOOST'
)

print("=== HUMAN REVIEW QUEUE PREVIEW (Top 10 High-Impression Flags) ===")
print(df_review[['content_hash_id', 'total_impressions', 'avg_position', 'ctr', 'flag_reason', 'requires_human_signoff']])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== HUMAN REVIEW QUEUE PREVIEW (Top 10 High-Impression Flags) ===
            content_hash_id  total_impressions  avg_position       ctr  \
0  content_755d951187fcd70a             1858.0      1.854929  0.003229   
1  content_c03ecafd4c999f15            10849.0      8.240351  0.002028   
2  content_df22bda1218f13ff             2099.0      3.066796  0.000476   
3  content_99d1bfa046d715ee            27871.0      4.829724  0.001650   
4  content_3ff938461f2eae39             7922.0      3.673689  0.002903   
5  content_b4de71c8ef5c4791             3535.0      2.951764  0.007072   
6  content_d7568011c4325a33             1558.0      5.264774  0.001926   
7  content_0b2aa5e8424bb65b             1414.0      4.515413  0.000707   
8  content_e847a4dcc8af3742             2021.0      7.653508  0.000000   
9  content_85b1be9944e4e19d             1257.0      6.758359  0.000000   

               flag_reason  requires_human_signoff  
0  PAGE_1_CTR_OPTIMIZATION                    True  
1  PAGE_1_CTR

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Monitoring and Retrain Triggers

The recommendations provided by this playbook go stale under specific environmental shifts. The model and playbook must be re-evaluated when any of the following triggers fire:

1. **Google Core Algorithm Update Trigger:** A major Search ranking update occurs, altering macro position-to-CTR benchmarks across the industry.
2. **Portfolio Baseline CTR Decay Trigger:** Aggregate portfolio CTR drops by >15% relative to the 90-day moving baseline.
3. **Data Staleness Trigger:** The underlying dataset passes 30 calendar days without a fresh monthly snapshot update.
4. **Feature Distribution Drift Trigger:** More than 20% of active client sites experience drastic impression changes due to site migrations or domain-level updates.

In [5]:
# Monitoring Check Script
current_avg_ctr = df_review['ctr'].mean()
historical_ctr_threshold = 0.015  # 1.5% portfolio CTR baseline

print("=== PLAYBOOK HEALTH & MONITORING AUDIT ===")
print(f"Current Validation Portfolio Mean CTR: {current_avg_ctr:.4f}")
print(f"Historical Threshold Baseline: {historical_ctr_threshold:.4f}")

if current_avg_ctr < historical_ctr_threshold:
    print("STATUS: ALERT - Portfolio CTR below threshold. Trigger model re-calibration.")
else:
    print("STATUS: HEALTHY - Model performance within normal operating bounds.")

=== PLAYBOOK HEALTH & MONITORING AUDIT ===
Current Validation Portfolio Mean CTR: 0.0020
Historical Threshold Baseline: 0.0150
STATUS: ALERT - Portfolio CTR below threshold. Trigger model re-calibration.


### 5. Exports for the Paper
The cell below generates all final receipts for the Week 8 Capstone paper:
1. `work/outputs/action_playbook_queue.csv` — The prioritized recommendation queue.
2. `work/outputs/w07_playbook_metrics.json` — Metrics summary receipt.
3. `work/figures/action_distribution.png` — Distribution chart for paper inclusion.

In [4]:
import duckdb
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata

# 0. Initialize DuckDB Connection with HF Secret Token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Ensure output and figure directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

# 1. Generate Complete Scored & Categorized Playbook Queue
full_playbook_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) AS total_clicks,
        CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) ELSE 0 END AS ctr
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id
""").df().fillna(0)

# Apply Rule / Model Priority Logic
conditions = [
    (full_playbook_df['avg_position'] >= 4) & (full_playbook_df['avg_position'] <= 10) & (full_playbook_df['ctr'] < 0.02),
    (full_playbook_df['avg_position'] > 10) & (full_playbook_df['avg_position'] <= 20),
    (full_playbook_df['avg_position'] > 20) & (full_playbook_df['total_impressions'] > 500)
]

choices_reason = ['HIGH_IMPRESSION_LOW_CTR', 'PAGE2_HIGH_POTENTIAL', 'POSITION_DECAY_STALE']
choices_action = ['REWRITE_TITLE_AND_META', 'ADD_INTERNAL_LINKS', 'REFRESH_CONTENT']

full_playbook_df['reason_code'] = np.select(conditions, choices_reason, default='MAINTAIN_MONITOR')
full_playbook_df['action_label'] = np.select(conditions, choices_action, default='NO_ACTION_REQUIRED')

# Score formula
full_playbook_df['priority_score'] = (
    (full_playbook_df['total_impressions'] / (full_playbook_df['avg_position'] + 1)) *
    (1 - full_playbook_df['ctr'])
)

ranked_playbook = full_playbook_df.sort_values(by='priority_score', ascending=False).reset_index(drop=True)

# Export Queue CSV
csv_path = 'work/outputs/action_playbook_queue.csv'
ranked_playbook.to_csv(csv_path, index=False)
ranked_playbook.to_csv('../outputs/action_playbook_queue.csv', index=False)
print(f"1. Exported Playbook Queue CSV to {csv_path} ({len(ranked_playbook)} rows)")

# 2. Export Metrics JSON Receipt
metrics_data = {
    "total_content_items_evaluated": int(len(ranked_playbook)),
    "actionable_flags_count": int((ranked_playbook['action_label'] != 'NO_ACTION_REQUIRED').sum()),
    "rewrite_title_meta_count": int((ranked_playbook['action_label'] == 'REWRITE_TITLE_AND_META').sum()),
    "add_internal_links_count": int((ranked_playbook['action_label'] == 'ADD_INTERNAL_LINKS').sum()),
    "refresh_content_count": int((ranked_playbook['action_label'] == 'REFRESH_CONTENT').sum()),
    "pipeline_status": "EXPORTS_VERIFIED_COMPLETE"
}

json_path = 'work/outputs/w07_playbook_metrics.json'
with open(json_path, 'w') as f:
    json.dump(metrics_data, f, indent=4)
with open('../outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics_data, f, indent=4)
print(f"2. Exported Metrics JSON Receipt to {json_path}")

# 3. Generate & Save Action Distribution Figure
plt.figure(figsize=(8, 4.5))
action_counts = ranked_playbook['action_label'].value_counts()
action_counts.plot(kind='barh', color='#2b5c8f')
plt.title('Content Action Playbook Queue Distribution')
plt.xlabel('Number of URLs Flagged')
plt.ylabel('Action Archetype')
plt.tight_layout()

fig_path = 'work/figures/action_distribution.png'
plt.savefig(fig_path, dpi=300)
plt.savefig('../figures/action_distribution.png', dpi=300)
plt.close()
print(f"3. Saved Action Distribution Chart to {fig_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Exported Playbook Queue CSV to work/outputs/action_playbook_queue.csv (331437 rows)
2. Exported Metrics JSON Receipt to work/outputs/w07_playbook_metrics.json
3. Saved Action Distribution Chart to work/figures/action_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.